In [1]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques
Fichiers copiés dans : data


In [10]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [3]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [24]:
test_ids = df_test['Id'].copy()

In [ ]:
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

In [65]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

0.12457570576748891


In [11]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

modeles = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10.0),
    'Lasso':            Lasso(alpha=0.001),
    'ElasticNet':       ElasticNet(alpha=0.001, l1_ratio=0.5),
    'RandomForest':     RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'KNN':              KNeighborsRegressor(n_neighbors=10),
}

In [74]:

for key, mod in modeles.items():
    mod.fit(X_train, np.log1p(y_train))
    y_pred = mod.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
    print(f"{key:20s} : {rmse:.4f}")


LinearRegression     : 0.1246
Ridge                : 0.1286
Lasso                : 0.1324
ElasticNet           : 0.1259


KeyboardInterrupt: 

In [12]:
from sklearn.model_selection import cross_val_score
import sklearn.metrics
sklearn.metrics.get_scorer_names()
print(sklearn.metrics.get_scorer_names()
)

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'd2_brier_score', 'd2_log_loss_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_m

In [68]:
X,_,_=p.preprocessing(X)

In [76]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

for key, mod in modeles.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('mod', mod)])
    scores = cross_val_score(pipe, X, np.log1p(y), cv=5, scoring='neg_mean_squared_error')
    print(f"{key:20s} : {np.sqrt(-scores.mean()):.4f}")

LinearRegression     : 0.1393
Ridge                : 0.1369
Lasso                : 0.1323
ElasticNet           : 0.1342
RandomForest         : 0.1424
GradientBoosting     : 0.1254
KNN                  : 0.1979


In [13]:
from preprocessing import ordinal_scales   # si tu l'y laisses, hors de la fonction

In [14]:
ordinal_cols = list(ordinal_scales.keys())

num_cols = X.select_dtypes(include='number').columns.tolist()

# toutes les colonnes texte/bool, MOINS les ordinales
cat_cols = X.select_dtypes(include=['bool', 'object']).columns.tolist()
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

num_cols = [c for c in num_cols if c != 'Id']

In [15]:
print("ordinal :", len(ordinal_cols), ordinal_cols)
print("nominal :", len(nominal_cols), nominal_cols)
print("num     :", len(num_cols), num_cols)

ordinal : 19 ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual', 'BsmtQual', 'GarageQual', 'GarageCond', 'FireplaceQu', 'PoolQC', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish', 'Utilities', 'LotShape', 'LandSlope', 'PavedDrive', 'Functional', 'Fence']
nominal : 24 ['MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtCond', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition']
num     : 36 ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageAre

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, PowerTransformer

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
])

ordinal_pipe= Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
('OrdinalEncoder',OrdinalEncoder(categories=[ordinal_scales[c] for c in ordinal_cols]))])

nominal_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),('OneHotEncoder',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])


In [17]:
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('ord', ordinal_pipe, ordinal_cols),
    ('nom', nominal_pipe, nominal_cols),
])

In [19]:
for key, mod in modeles.items():
    pipe = Pipeline([('preprocessor',preprocessor), ('mod', mod)])
    scores = cross_val_score(pipe, X, np.log1p(y), cv=10, scoring='neg_mean_squared_error')
    print(f"{key:20s} : {np.sqrt(-scores.mean()):.4f}")

LinearRegression     : 0.1377
Ridge                : 0.1285
Lasso                : 0.1264
ElasticNet           : 0.1258
RandomForest         : 0.1408
GradientBoosting     : 0.1236
KNN                  : 0.1735


In [ ]:
model_pipe=Pipeline([('preprocessor',preprocessor), ('Ridge', modeles['Ridge'])])

In [20]:
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Ridge()),          # pas besoin d'alpha ici
])

param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 200.0]}

grid = GridSearchCV(ridge_pipe, param_grid, cv=5,
                    scoring='neg_mean_squared_error')
grid.fit(X, np.log1p(y))

print("meilleur alpha :", grid.best_params_)
print("RMSE           :", np.sqrt(-grid.best_score_))

meilleur alpha : {'model__alpha': 10.0}
RMSE           : 0.12947232608498913


In [22]:
lasso_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Lasso(max_iter=10000)),   # max_iter élevé pour éviter le warning de convergence
])

param_grid_lasso = {'model__alpha': [0.0001, 0.001, 0.005, 0.01, 0.05]}


grid = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5,
                    scoring='neg_mean_squared_error')
grid.fit(X, np.log1p(y))

print("meilleur alpha :", grid.best_params_)
print("RMSE           :", np.sqrt(-grid.best_score_))

meilleur alpha : {'model__alpha': 0.001}
RMSE           : 0.125892708656521


In [25]:
# grid a déjà été fit sur X. On prédit directement sur df_test.
predictions_log = grid.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission.csv', index=False)
print(soumission['SalePrice'].describe())
print(soumission['SalePrice'].isna().sum())   # doit être 0

count      1459.000000
mean     177704.899306
std       73029.976137
min       46592.085479
25%      127243.349219
50%      158574.137577
75%      211169.552997
max      611958.748641
Name: SalePrice, dtype: float64
0


In [27]:
Boost_pipe=Pipeline([('preprocessing',preprocessor),('GradientBoosting',GradientBoostingRegressor(random_state=42))])
Boost_pipe.fit(X, np.log1p(y)) 
predictions=np.expm1(Boost_pipe.predict(df_test))
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission.csv', index=False)